<a href="https://colab.research.google.com/github/20020109tharindu/ML_Final_Project/blob/svm/models/svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Shakya

## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Clone GitHub repo
!git clone https://github.com/20020109tharindu/ML_Final_Project.git
%cd ML_Final_Project

Cloning into 'ML_Final_Project'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 63 (delta 28), reused 39 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 208.22 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/ML_Final_Project


## Install Libraries

In [3]:
# Cell 3 — Install libraries
!pip install imbalanced-learn -q

## Import Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## Load Cleaned Data from Google Drive

In [6]:
DATA_PATH = '/content/drive/MyDrive/FraudDetection/cleaned_data/'

X_train = pd.read_csv(DATA_PATH + 'X_train.csv')
X_test  = pd.read_csv(DATA_PATH + 'X_test.csv')
y_train = pd.read_csv(DATA_PATH + 'y_train.csv').squeeze()
y_test  = pd.read_csv(DATA_PATH + 'y_test.csv').squeeze()

print('✅ Data loaded successfully!')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   Fraud cases in train : {y_train.sum():,}')
print(f'   Fraud cases in test  : {y_test.sum():,}')

✅ Data loaded successfully!
   X_train : (454902, 30)
   X_test  : (56962, 30)
   Fraud cases in train : 227,451
   Fraud cases in test  : 98


## Build & Train the SVM Model

In [11]:
# 1. Configuration
SAMPLE_SIZE = 30000   # increase to 50000 if you have time

# 2. Balanced Sampling Fix
# We take exactly 15,000 from each class to reach 30,000 total
# fraud_idx = y_train[y_train == 1].sample(n=SAMPLE_SIZE // 2, random_state=42).index
# legit_idx = y_train[y_train == 0].sample(n=SAMPLE_SIZE // 2, random_state=42).index
# fraud_idx = y_train[y_train == 1].sample(n=5000, random_state=42).index
# legit_idx = y_train[y_train == 0].sample(n=25000, random_state=42).index
fraud_idx = y_train[y_train == 1].sample(n=2000, random_state=42).index
legit_idx = y_train[y_train == 0].sample(n=40000, random_state=42).index
sample_idx = fraud_idx.tolist() + legit_idx.tolist()

X_sample = X_train.loc[sample_idx]
y_sample = y_train.loc[sample_idx]

print(f'Training on {len(X_sample):,} samples')
print(f'Fraud : {y_sample.sum():,}')
print(f'Legit : {(y_sample == 0).sum():,}')

Training on 42,000 samples
Fraud : 2,000
Legit : 40,000


In [12]:
# 3. Build Model Pipeline
# SVM is distance-based, so StandardScaler is REQUIRED for convergence speed.
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=0.1,             # Low C for smoother boundary
        gamma=0.01,        # Specific gamma to prevent 'bubbles'
        class_weight=None, # Remove this since we have 40k legit samples now
        probability=True,
        random_state=42
    ))
])

# 4. Train
print('\nTraining SVM Pipeline (Expected time: 2-5 minutes)...')
start_time = time.time()

svm_pipeline.fit(X_sample, y_sample)

train_time = time.time() - start_time
print(f'✅ Model trained in {train_time:.1f} seconds!')

# 5. Evaluation
# Use the pipeline to predict; it will handle scaling X_test automatically
y_pred = svm_pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Training SVM Pipeline (Expected time: 2-5 minutes)...
✅ Model trained in 25.5 seconds!

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.77      0.79      0.78        98

    accuracy                           1.00     56962
   macro avg       0.88      0.89      0.89     56962
weighted avg       1.00      1.00      1.00     56962



## Make Predictions

In [13]:
# Predict using the pipeline (it handles scaling X_test automatically)
y_pred       = svm_pipeline.predict(X_test)
y_pred_proba = svm_pipeline.predict_proba(X_test)[:, 1]

print('✅ Predictions done with Optimized Pipeline!')
print(f'   Predicted Fraud : {y_pred.sum():,}')
print(f'   Actual Fraud    : {y_test.sum():,}')

✅ Predictions done with Optimized Pipeline!
   Predicted Fraud : 100
   Actual Fraud    : 98
